# Dataset Audit — is this dataset right for my project?

Run this notebook **once per candidate dataset**, top to bottom, cell by cell. Each check prints
*why it matters*, *what it found*, and a traffic light:

| | meaning |
|---|---|
| ✅ PASS | fine |
| ⚠️ WARN | usable, but handle it or name it as a limitation |
| ❌ FAIL | blocker for this task |
| ℹ️ INFO | fact worth knowing |
| 📝 TODO | only you can find this out (paper, website, licence) |

At the end you get a **verdict** and a **dataset card** (`.md` + `.json`). The last cell compares all
datasets you have checked side by side.

**What gets checked**

| Part | Question | Why it matters |
|---|---|---|
| A. Origin & licence | Who made it, may I use it, how do I cite it? | No licence = no permission. Unknown origin = unknown bias. |
| B1. Files & size | What formats, how many GB, any documentation? | Feasibility (Colab disk, download time). |
| B2. Size & resolution | Image size, pixel size in mm, slices, number of samples | Fixed input size, physical geometry, enough data to validate. |
| B3. Pixel values | HU, 8-bit or normalised? NaNs, blank images? | Decides what is physically meaningful. |
| B4. Visual check | Does it look right? | Artefacts, burned-in text, cropped field of view. |
| B5. Patients & splits | How many patients, is there leakage? | Same patient in train+test = fake good results. |
| B6. Labels & balance | Classes, masks, imbalance | 95% one class → accuracy is meaningless. |
| B7. Diversity | Scanners, sites, sex, age | Generalisation and bias. |
| B8. Duplicates | Exact and near copies | Inflated results, smaller real dataset. |
| B9. Privacy | Names/birth dates in DICOM tags? | Legal; never push such data to GitHub/public Colab. |
| C. Task fit | Does it fit *my* task? | A good dataset can still be the wrong one. |

> Tip: you don't need the whole dataset. Download **5–10 patients** first, audit them, then decide.

## 0. Setup

In [ ]:
# Install readers (only needed once per environment; harmless if already installed)
%pip install -q pydicom SimpleITK h5py pillow pandas matplotlib

In [ ]:
# dataset_audit.py must sit next to this notebook.
# In Google Colab: upload it with the file browser on the left, or run this cell and pick the file.
import os, sys
if "google.colab" in sys.modules and not os.path.exists("dataset_audit.py"):
    from google.colab import files
    files.upload()

import pandas as pd
pd.set_option("display.max_colwidth", None)
import dataset_audit as da
da.available_readers()

In [ ]:
# Alternative Dataset: Brain CT npy-1
# use output file path for DATA_ROOT

import kagglehub

# Download latest version
path = kagglehub.dataset_download("josepc/brain-ct-npy-1")

print("Path to dataset files:", path)

In [ ]:
# Alternative Dataset: PhantomX
# use output file path for DATA_ROOT

import kagglehub

# Download latest version
path = kagglehub.dataset_download("phantomxdata/head-ct-collection-phantomx")

print("Path to dataset files:", path)

## 1. Configuration — the only cell you need to edit

* `DATA_ROOT`: folder with the (sample of the) dataset. `None` = synthetic demo dataset with planted problems.
* `TASK`: `"sparse_view_ct"`, `"reconstruction"`, `"segmentation"`, `"classification"` or `"other"`.
* `PATIENT_ID_REGEX`: how to find the patient ID in the file path if it isn't in DICOM tags,
  e.g. `r"(P\d{3})"` for `P001_slice00.png`, `r"(LIDC-IDRI-\d{4})"`, `r"patient[_-]?(\d+)"`. The first group is the ID.
* `LABELS_FROM`: `None` (no labels / reconstruction), `"folders"` (class = parent folder name), or
  `("table", "path/to/labels.csv", "id_column", "label_column", "stem" or "patient")`.

In [ ]:
DATASET_NAME     = "Demo dataset"
DATA_ROOT        = None                 # e.g. "/content/drive/MyDrive/datasets/LoDoPaB_sample"
TASK             = "sparse_view_ct"
PATIENT_ID_REGEX = None                 # demo: r"(P\d{3})"
LABELS_FROM      = None                 # reconstruction: keep None
MIN_INPLANE_PX   = 256                  # smallest image side you'd accept
MASK_PATTERN     = r"(?:mask|seg|label|gt|annot)"   # segmentation only: how mask files are named

if DATA_ROOT is None:                   # demo mode
    DATA_ROOT = da.make_demo_dataset("demo_dataset")
    PATIENT_ID_REGEX, LABELS_FROM = r"(P\d{3})", "folders"

rep = da.Report(DATASET_NAME, TASK)

## Part A — Desk check (do this *before* downloading)

Fill in what you find on the dataset website and in its paper. Leave unknown fields empty: they show up as 📝 TODO.
Typical places: Zenodo / TCIA / PhysioNet / grand-challenge.org / Kaggle page, the *data descriptor* paper,
the GitHub issues of the dataset.

In [ ]:
provenance = {
    "source_url":         "",
    "publisher":          "",
    "paper_to_cite":      "",
    "version_and_date":   "",
    "licence":            "CC BY-NC-ND 4.0",   # e.g. CC BY 4.0 / CC0 / TCIA Restricted / custom DUA / none
    "access":             "open",              # open / registration / credentialed / DUA
    "ethics_approval":    "",
    "de_identification":  "",
    "acquisition":        "",                  # scanners, sites, years, dose/protocol
    "population":         "",                  # who: age, sex, disease, inclusion criteria
    "label_source":       "not needed (reconstruction)",
    "raw_data_available": "",                  # sinograms/projections? or only images?
    "official_split":     "",
    "known_issues":       "",
}
da.check_provenance(provenance, rep)

## Part B — Automatic checks on the files

### B1. Files & size

In [ ]:
inv = da.inventory(DATA_ROOT, rep)
inv.head()

In [ ]:
# Read headers (metadata only, fast) and group files into samples (DICOM slices -> one series/volume)
meta  = da.read_headers(inv, rep, patient_id_regex=PATIENT_ID_REGEX)
items = da.build_items(meta)

masks = None
if TASK == "segmentation":
    items, masks = da.split_images_masks(items, MASK_PATTERN)

items[["item_id", "kind", "shape", "spacing", "dtype", "patient_id", "split"]].head(10)

### B2. Size & resolution

In [ ]:
da.check_size_resolution(items, rep, min_inplane=MIN_INPLANE_PX)

### B3. Pixel values

In [ ]:
pixel_stats = da.check_pixels(items, rep, n=30)

### B4. Look at the data
Change `window=(level, width)` for CT in HU: soft tissue `(40, 400)`, bone `(400, 1800)`, lung `(-600, 1500)`.
Look for: artefacts (metal streaks, motion), text burned into the image, cut-off anatomy, wrong orientation, empty images.

In [ ]:
da.show_samples(items, n=12, window=None)

### B5. Patients & splits (data leakage)

In [ ]:
da.check_patients(items, rep)

### B6. Labels & class balance

In [ ]:
if TASK in ("sparse_view_ct", "reconstruction"):
    print("Reconstruction task: no labels needed, the full-view image is the target. Section skipped.")
elif TASK == "segmentation":
    da.check_masks(items, masks, rep, mask_pattern=MASK_PATTERN)
elif LABELS_FROM == "folders":
    labels = da.labels_from_folders(items)
    da.check_labels(labels, rep)
elif isinstance(LABELS_FROM, tuple) and LABELS_FROM[0] == "table":
    _, path, id_col, label_col, match = LABELS_FROM
    labels = da.labels_from_table(items, path, id_col, label_col, match=match)
    da.check_labels(labels, rep)
else:
    print("No labels configured (fine for reconstruction: the full-view image is the target).")

### B7. Balance & diversity of the metadata
Counted per patient. If you have a clinical/metadata CSV with a `patient_id` column, pass it as `extra=pd.read_csv(...)`.

In [ ]:
da.check_balance(items, rep)   # optionally: extra=pd.read_csv('clinical.csv')

### B8. Duplicates

In [ ]:
da.check_duplicates(items, rep)

### B9. Privacy

In [ ]:
da.check_privacy(items, rep)

## Part C — Does it fit my task?

In [ ]:
da.check_task_fit(TASK, items, rep, pixel_stats=pixel_stats, provenance=provenance)

## Result

Sorted: blockers first. Then save the dataset card into `dataset_cards/`.
Your own notes (e.g. "chose this because…") can go into `rep.add("My notes", "INFO", "...")` before saving.

In [ ]:
rep.summary()

In [ ]:
rep.save("dataset_cards")

## Compare all datasets checked so far
Every saved card appears as one column. Good for the "dataset selection" part of a report.

In [ ]:
da.compare_reports("dataset_cards")